In [ ]:
while True:
    logging.info(f"正在抓取{start_index//result_per_page + 1}页|抓去取了{start_index}条bugs|截至时间为{pub_end_date}")
    params ={
             "startIndex":start_index,
             "resultsPerPage":result_per_page,
             "pubStartDate":pub_start_date,
             "pubEndDate":pub_end_date
    }
    response = None

    if response.status_code == 200:#这表示服务器访问顺利
        try:
            data = response.json()
            #将返回的数据里的所有漏洞放进bugs这个列表
            bugs = data.get("vulnerabilities",[])

            if not bugs:
                break

            #将所有漏洞都循环一遍
            for bug in bugs:
                bug_datas = bug.get("cve",{})#如果存在漏洞，取出第一个漏洞，并且命名为bug_bags
                bug_name = bug_datas.get("id","")

                metrics = bug_datas.get("metrics",{})#将bug_bags这个漏洞所有信息放进metrics这个列表里面

                #用以上的评分系统进行抓取信息，将评分系统在漏洞信息列表得出的所需数据做成一个新的列表
                for cvss_true_type in cvss_types:
                    metrics_list = metrics.get(cvss_true_type,[])

                    #取评分系统的第一版所需数据
                    if metrics_list:
                        metrics_data = metrics_list[0].get("cvssData",{})
                        break

                    #提出描述
                descriptions = bug_datas.get("descriptions",[])
                target_desc = get_best_descriptions(descriptions,target_lang="en")

                all_result.append({
                    "cve_id":bug_datas.get("id",bug_name),
                    "source":"NVD",
                    "descriptions":target_desc or "",
                    "cvss_score":cvss_score,
                    "cvss_version":cvss_version or "",
                    "pubStartDate":pub_start_date,
                    "pubEndDate":pub_end_date
                })
            if len(bugs) < result_per_page or page >= max_page:
                break
            start_index += result_per_page
            page += 1
            time.sleep(2)

        except ValueError as error:
            logging.error("无法将数据统计为Json格式")
            break

    else:
        logging.error(f"服务器被拒绝访问或者服务器错误，状态码：{response.status_code}")
        break
